# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### RannableLambda
일반 python 함수를 lcel 체인에서 사용할 수 있는 Runnable 형태로 wrapping 처리해주는 클래스

In [ ]:
# 입력을 받아 내장된 함수를 실행하는 Runnable
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕 만나서 반갑다')

10

In [ ]:
# batch : 여러 건의 입력을 일괄처리해줌
runnable.batch(['안녕 만나서 반갑다', '너도? 나도', '?!', '😂'])

[10, 6, 2, 1]

In [4]:
def celsius_to_fahrenheit(celsius):
    return celsius*9 / 5+32

celsius_temps = [0, 25, 100, -10, 37]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temps)

[32.0, 77.0, 212.0, 14.0, 98.6]

In [ ]:
import time # 출력 딜레이용

def generator(x):
    for y in x: # 입력을 문자 단위로 순회
        yield y # 한 글자씩 반환 (스트리밍 방식)
runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~'):
    print(chunk, end='',flush=True) # flush 즉시 출력
    time.sleep(0.1) # 글자 출력마다 딜레이 0.1초

안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒안녕하세요~😊😂🤣😒

In [26]:
# 사용 예시
def gen(x):
    for y in x:
        yield y

gen10 = gen(range(10))
gen10
for i in gen10:
    print(i)

0
1
2
3
4
5
6
7
8
9


In [ ]:
# gen10 = gen(range(10))
next(gen10) # 제너레이터 다음 값 1개 반환 (다 꺼내고나면 StopIteration 발생)

0

### RunnableSequence
Runnable 객체를 순차열결해주는 Runnable 객체

In [ ]:
from langchain_core.runnables import RunnableSequence # Runnable들을 순서대로 연결하는 시퀀스

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableSequence(runnable1, runnable2)
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

In [43]:
chain = runnable1 | runnable2
chain.invoke(3)

[{'foo': 3}, {'foo': 3}, {'foo': 3}]

### RunnableParallel
여러 Runnable 객체를 인자로 받아, 병렬처리 후 각각의 응답을 하나의 dict로 반환

In [49]:
from langchain_core.runnables import RunnableParallel # 여러 Runnable들을 같은 입력으로 병렬로 실행

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: [x] * 3)

chain = RunnableParallel(r1= runnable1, r2= runnable2)
chain.invoke(3)

{'r1': {'foo': 3}, 'r2': [3, 3, 3]}

- 사용자가 준 주제를 이용해서 삼행시, 농담, 시를 각각 생성해서 하나의 응답으로 반환

In [ ]:
from langchain_core.prompts import PromptTemplate    # prompt chain 구성
from langchain.chat_models import init_chat_model    # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환
from langchain_core.runnables import RunnableParallel

llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시의 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
n_poem_chain = acrostic_poem_prompt | llm | output_parser
joke_prompt = PromptTemplate.from_template(
    '당신은 한국식 농담계의 고수입니다. 다음 주제로 배꼽이 빠질만한 농담을 지어주세요. 주제 : {topic}'
)
joke_chain = joke_prompt | llm | output_parser
poem_prompt = PromptTemplate.from_template(
    '당신은 현대시 작가 입니다. 다음 주제로 눈물이 나올 정도의 감성적인 시를 지어주세요. 주제 : {topic}'
)
poem_chain = poem_prompt | llm | output_parser

# 동일 입력(topic)으로 3개 체인을 병렬 실행 {acrostic_poem: 실행결과, ...}
chain = RunnableParallel(acrostic_poem=n_poem_chain,joke=joke_chain,poem=poem_chain)

def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem  = input_dict['poem']
    return f"""
    n행시 :
    {acrostic_poem}

    농담 :
    {joke}

    현대시 :
    {poem}
    """

chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic': '강아지'}))


    n행시 :
    강: 강아지의 꼬리가 살랑살랑 흔들리면  
아: 아무리 힘든 하루도 웃음이 피어나고  
지: 지금 이 순간, 세상에서 제일 행복해져!

    농담 :
    강아지가 제일 싫어하는 도시는?

**개나리?**  
아니요, **개인적인 일이 많은 ‘개인도시’요.**  
맨날 “개인기 보여줘!” 해서 너무 피곤하대요. 🐶

---

강아지가 수학시험에서 100점을 받았대요.  
비결을 물어보니…

**“문제를 보자마자 다 풀었지. 나는 원래 ‘개념’이 확실하거든!”**

---

강아지가 미용실에 갔어요.

미용사가 물었습니다.  
“어떤 스타일로 해드릴까요?”

강아지가 대답했어요.  
**“오늘은 좀… 개성 있게요.”**

---

강아지가 가장 좋아하는 음료는?

**멍물!**  
사람들이 물을 마실 때 강아지는 옆에서  
“멍… 물 좀 더 주세요!” 🐾

---

강아지가 인터넷에 처음 접속했어요.  
검색창에 제일 먼저 친 말은?

**“개껌 할인”**

---

강아지가 주인에게 혼났어요.

“왜 신발을 물어뜯었어?”

강아지가 억울한 표정으로 말했어요.  
**“저는 신발을 먹은 게 아니라… 발음 연습을 한 거예요. ‘씹어’ 발음이 어려워서!”**

---

강아지가 노래방에 가서 부른 노래는?

**〈개똥벌레〉**

그런데 너무 감정이입해서  
후렴구마다 **“멍~ 멍~ 멍멍멍~”** 하고 울었다고 합니다.

    현대시 :
    ### 네가 떠난 뒤에도

네가 떠난 뒤에도  
현관 앞 신발은 두 켤레다.

한 짝은 내가 신고 나갈 것이고  
다른 한 짝은  
네가 꼬리를 흔들며 달려오던 날들을  
가만히 기억하고 있다.

아침이면 습관처럼  
사료 그릇에 물을 붓는다.  
텅 빈 그릇이  
세상에서 가장 조용한 울음이라는 걸  
나는 이제야 알았다.

네가 눕던 자리의 털 한 올을  
며칠 동안 치우지 못했다.  
그 작은 흔적이 사라지면  
정말 네가 이 집에서  
한 번도 살지 않았던 사람이 될까 봐.

너는 말이

### RunnablePassThorugh
- 사용자의 입력값을 그대로 전달해주는 Runnable

In [53]:
from langchain_core.runnables import RunnablePassthrough
prompt = PromptTemplate.from_template(
    '당신은 n행시의 고수입니다. 다음 주제로 n행시를 지어주세요. 주제 : {topic}'
)
chain = {'topic': RunnablePassthrough()}| prompt | llm | output_parser

print(chain.invoke('학원수업'))

**학**: 학교 끝나자마자 달려온 학원,  
**원**: 원하는 꿈을 향해 한 걸음 더!  
**수**: 수업은 열심히, 질문은 당당하게,  
**업**: 업그레이드된 실력으로 꿈을 이룬다!


In [ ]:
prompt = PromptTemplate.from_template("""
    당신은 {n}행시의 고수입니다. 다음 주제로 {n}행시를 지어주세요. 
    주제 : {topic}
    출력형식
    ===== <주제> <n행시> =====
    <n행시 작성>
"""
)
chain = ({'topic': RunnablePassthrough()}
        | RunnablePassthrough.assign(     # 기존 topic만 있던 dict -> 새 key를 추가
            n= lambda x: len(x['topic']), # n = topic 길이
            k= lambda x:100
        ) 
        | prompt # 확장된 dict(topic, n, k)를 프롬프트에 주입해서 완성
        | llm
        | output_parser
)

print(chain.invoke('아이스크림'))

===== 아이스크림 5행시 =====  
아무리 더운 여름날에도  
이 한 입이면 마음이 사르르 녹고  
스르르 퍼지는 달콤한 행복에  
크게 웃음이 피어나며  
림처럼 둥근 기쁨이 가득해진다
